# GenAI Pipeline — Patents

LLM-based screening of patents for end product classification.
Uses Claude with structured output via the Anthropic Python SDK.

Adapted from the publications GenAI pipeline.

### 1. Imports and Configuration

In [24]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator, create_model
from typing import Literal

load_dotenv("../../.env")

DB_PATH = "../../patents_training.db"
OUTPUT_DIR = Path(".")

### 2. Data Inspection

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df())

df = con.sql("SELECT * FROM patents_raw").df()   # ← UPDATE: table name
con.close()

df = df[df['scope'] == 'in']
df['endproduct'] = df['endproduct'].fillna('Agnostic')

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nScope distribution:")
print(df["scope"].value_counts())
print(f"\nEnd product distribution:")
print(df["endproduct"].value_counts())
df.head()

Tables:
                 name
0  patents_embeddings
1         patents_raw

Shape: (1275, 15)

Columns: ['id', 'family_id', 'application_number', 'title', 'abstract', 'cpc', 'publication_year', 'jurisdiction', 'scope', 'pillar', 'subpillar', 'research_category', 'endproduct', 'ingredient', 'truncated']

Scope distribution:
scope
in    1275
Name: count, dtype: int64

End product distribution:
endproduct
Meat                                      516
Milk and milk proteins                    234
Agnostic                                  210
Cross-cutting                             110
Cheese                                     83
Yoghurt and fermented dairy                44
Eggs and egg proteins                      24
Chocolate, desserts, and confectionery     18
Cream and ice cream                        17
Fish and seafood                           15
Spreads, sauces, and condiments             4
Name: count, dtype: int64


,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
2,US-20170298457-A1,42829610,US15360298,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12N9/1205', 'A23C2220/206', ...",2017,US,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
3,EP-2473058-A1,42829610,EP10751643A,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12R2001/46', 'C12Y207/01006'...",2012,EP,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
28,EP-4424172-A3,52462090,EP24182765.8,A PROTEINACEOUS MEAT ANALOGUE HAVING AN IMPROV...,The invention concerns an extended shelf-life ...,"['A23V2002/00', 'A23V2200/20', 'A23J3/22', 'A2...",2025,EP,in,PB,NaN,End product formulation,Meat,NaN,False
29,US-11666069-B2,52462090,US17530304,Proteinaceous meat analogue having an improved...,An extended shelf-life proteinaceous meat anal...,"['A23J3/22', 'A23J3/18', 'A23V2200/20', 'A23V2...",2023,US,in,PB,NaN,End product formulation,Meat,NaN,False
32,EP-4627930-A3,52596486,EP25169128.3,VARIANTS OF CHYMOSIN WITH IMPROVED MILK-CLOTTI...,Variants of chymosin with improved milk-clotti...,"['A23C19/04', 'A23C19/041', 'C12N9/6483', 'A23...",2026,EP,in,PB,NaN,End product formulation,Cheese,NaN,False


### 3. Balanced Subset Creation

Two test sets with balanced representation across scope and pillar:
- `initial_test_data`: ~14 records (6 out, 2 PB, 2 F, 2 cultivated, 2 cross-cutting)
- `test_data_100`: ~110 records (40 out, 30 PB, 15 F, 15 cultivated, 10 cross-cutting)

Adjust counts to match the actual distribution in your patents dataset.

In [3]:
RANDOM_STATE=4

def create_balanced_sample(df, category_counts, random_state=RANDOM_STATE):
    """
    category_counts: dict mapping end product -> n, e.g.
        {"Meat": 5, "Cheese": 3, "Other": 2}
    Categories with no matching rows are skipped with a warning.
    If n exceeds available rows, all available rows are taken (with a warning).
    """
    samples = []
    for cat, n in category_counts.items():
        subset = df[df["endproduct"] == cat]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        if n > available:
            print(f"  Warning: '{cat}' — requested {n} but only {available} available, taking all.")
            n = available
        samples.append(subset.sample(n=n, random_state=random_state))
    combined = pd.concat(samples, ignore_index=True)
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [4]:
test_data = create_balanced_sample(df, {
    'Yoghurt and fermented dairy': 10,
    'Meat': 10,
    'Cheese': 10,
    'Milk and milk proteins': 10,
    'Cross-cutting': 10,
    'Cream and ice cream': 10,
    'Chocolate, desserts, and confectionery': 10,
    'Eggs and egg proteins': 10,
    'Fish and seafood': 10,
    'Agnostic': 10,
    'Spreads, sauces, and condiments': 10,
}, random_state=RANDOM_STATE)
print(f"test_data: {test_data.shape}")

test_data: (104, 15)


### 4. Save Subsets to Excel

In [ ]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=OUTPUT_DIR):
    path = output_dir / filename
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(test_data, f"endproduct_test_data_rand{RANDOM_STATE}.csv")

Saved 104 records to endproduct_test_data_rand4.csv


In [59]:
# Read subset data from file
test_data = pd.read_csv(f"endproduct_test_data_rand{RANDOM_STATE}.csv")

In [60]:
test_data.value_counts('endproduct')

endproduct
Cross-cutting                             15
Agnostic                                  14
Chocolate, desserts, and confectionery    10
Cream and ice cream                       10
Yoghurt and fermented dairy                9
Fish and seafood                           9
Meat                                       9
Eggs and egg proteins                      7
Cheese                                     7
Dairy                                      6
Milk and milk proteins                     4
Spreads, sauces, and condiments            4
Name: count, dtype: int64

### 5. Load Prompt and Select Dataset

In [61]:
PROMPT_PATH = "prompt_endproduct_patents_work.md"   # ← UPDATE: path to current prompt version
PROMPT_VERSION = "v3"

# ← CHANGE THIS to switch between datasets
# Options: initial_test_data (14 records), test_data_100 (110 records)
DATASET = test_data
DATASET_STR = "test_data"

In [62]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)

You are an expert in alternative proteins and food technology.

Your task is to classify a patent on alternative proteins into an end product category based on its title and abstract.

Before assigning a category, identify the primary end product or application context that the patent is designed for. The specific categories below are reserved for patents that primarily target one type of alternative protein end product. Assign Cross-cutting, Dairy, or Agnostic only when the patent genuinely does not target a single specific product type.

IMPORTANT: Base your classification ONLY on what the title and abstract explicitly state about the intended end product application. Do NOT infer an end product from external knowledge about how an ingredient, organism, or technology is commonly used. Do NOT infer a product category from the physical form of an ingredient (beads, microcapsules, spheres, gels, pastes) — a delivery system or encapsulated ingredient with a flavoured filling is not autom

### 6. API Call with Structured Output

In [63]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model
# If using Opus, comment out TEMPERATURE below

MAX_TOKENS = 512         # max tokens in response
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0
RETRY_MAX_SECONDS = 90.0

REPETITIONS = 1          # number of full runs; increase to measure output variance

# ================================================================
# CHECKPOINT CONFIG
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

In [64]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

END_PRODUCT_CATS = [
    "Meat",
    "Fish and seafood",
    "Milk and milk proteins",
    "Yoghurt and fermented dairy",
    "Cheese",
    "Cream and ice cream",
    "Agnostic",
    "Chocolate, desserts, and confectionery",
    "Eggs and egg proteins",
    "Cross-cutting",
    "Spreads, sauces, and condiments",
    "Dairy",
]

def make_schema(cats, include_reasoning):
    cats_map = {c.lower(): c for c in cats}
    cat_type = Literal[*cats]

    class _Base(BaseModel):
        @field_validator("primary", "secondary", mode="before", check_fields=False)
        @classmethod
        def normalise_case(cls, v):
            if isinstance(v, str):
                return cats_map.get(v.lower(), v)
            return v

    fields = {"primary": (cat_type, ...), "secondary": (cat_type, ...)}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    return create_model("ClassificationSchema", __base__=_Base, **fields)

ClassificationSchema = make_schema(END_PRODUCT_CATS, INCLUDE_REASONING)
print(f"Schema built: {END_PRODUCT_CATS}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_patent(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output

Schema built: ['Meat', 'Fish and seafood', 'Milk and milk proteins', 'Yoghurt and fermented dairy', 'Cheese', 'Cream and ice cream', 'Agnostic', 'Chocolate, desserts, and confectionery', 'Eggs and egg proteins', 'Cross-cutting', 'Spreads, sauces, and condiments', 'Dairy']


### 7. Error Handling with Retry

In [65]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter

In [66]:
def classify_with_error_handling(row, system_prompt):
    pat_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_patent(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pat_id}: model returned no structured output")
                return {"id": pat_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()}
            output["id"] = pat_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pat_id}: {last_error}")
    return {"id": pat_id, "status": "api_error", "error": str(last_error)}

### 8. Checkpoint Helpers

In [67]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()

### 9. Run on Test Data

In [68]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df


Run 1 / 1
  [1/104] JP-2025522935-A
  [2/104] US-20250176578-A1
  [3/104] US-20250098715-A1
  [4/104] US-20250241331-A1
  [5/104] EP-4514142-A1
  [6/104] NL-2031656-B1
  [7/104] WO-2023099270-A1
  [8/104] EP-4333632-A1
  [9/104] EP-3599885-A1
  [10/104] CN-121532076-A
  [11/104] EP-4656060-A1
  [12/104] JP-2025519945-A
  [13/104] FI-131418-B1
  [14/104] EP-4754246-A1
  [15/104] US-20190110501-A1
  [16/104] US-20250331537-A1
  [17/104] CN-121908951-A
  [18/104] US-12635717-B2
  [19/104] WO-2023009007-A1
  [20/104] EP-4547033-A1
  [21/104] WO-2025262298-A1
  [22/104] EP-4753472-A1
  [23/104] EP-4529772-A3
  [24/104] WO-2025157817-A1
  [25/104] EP-4358738-A1
  [26/104] GB-2641787-A
  [27/104] EP-4482328-A1
  [28/104] JP-2025502978-A
  [29/104] EP-4593632-A1
  [30/104] JP-2022506260-A
  [31/104] WO-2025056708-A1
  [32/104] US-20240148016-A1
  [33/104] EP-4597070-A1
  [34/104] EP-4661693-A1
  [35/104] EP-4615237-A1
  [36/104] US-20250000117-A1
  [37/104] WO-2025233496-A1
  [38/104] EP-4444

,primary_LLM,secondary_LLM,reasoning_LLM,id,status,run
0,Milk and milk proteins,Agnostic,The patent describes a bioreactor apparatus an...,JP-2025522935-A,ok,1
1,Yoghurt and fermented dairy,Dairy,The patent explicitly describes a process for ...,US-20250176578-A1,ok,1
2,"Spreads, sauces, and condiments",Agnostic,The patent explicitly targets oat-based savour...,US-20250098715-A1,ok,1
3,Dairy,Yoghurt and fermented dairy,The patent describes methods for producing ani...,US-20250241331-A1,ok,1
4,Fish and seafood,Agnostic,The patent explicitly describes gel compositio...,EP-4514142-A1,ok,1
...,...,...,...,...,...,...
99,Fish and seafood,Cross-cutting,The patent explicitly describes a method of ma...,US-20250280859-A1,ok,1
100,Cross-cutting,Agnostic,This patent describes a food-use enzyme compos...,WO-2025233389-A1,ok,1
101,"Chocolate, desserts, and confectionery",Milk and milk proteins,The patent explicitly describes a chocolate pr...,EP-4633378-A1,ok,1
102,Cream and ice cream,Dairy,The patent explicitly targets vegan/non-dairy ...,EP-4498836-A1,ok,1


In [69]:
result_cols = ["id", "run", "primary_LLM", "secondary_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "pillar", "endproduct"]].merge(
    results_df[result_cols], on="id", how="left"
)

comparison["primary_correct"] = comparison["endproduct"] == comparison["primary_LLM"]
comparison["either_correct"]  = (
    (comparison["endproduct"] == comparison["primary_LLM"]) |
    (comparison["endproduct"] == comparison["secondary_LLM"])
)

# Overall metrics
n = len(comparison)
print(f"Top-1 accuracy (primary match):  {comparison['primary_correct'].mean():.0%}  (n={n})")
print(f"Top-2 accuracy (either match):   {comparison['either_correct'].mean():.0%}  (n={n})")

# Per-category breakdown
cat_stats = (
    comparison.groupby("endproduct")
    .agg(
        n=("primary_correct", "count"),
        primary_correct=("primary_correct", "sum"),
        top2_correct=("either_correct", "sum"),
    )
    .assign(
        primary_acc=lambda d: (d["primary_correct"] / d["n"]).map("{:.0%}".format),
        top2_acc=lambda d: (d["top2_correct"] / d["n"]).map("{:.0%}".format),
    )
)
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "pillar", "endproduct",
                "primary_LLM", "secondary_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["primary_correct", "either_correct"]
comparison[display_cols]

Top-1 accuracy (primary match):  93%  (n=104)
Top-2 accuracy (either match):   99%  (n=104)


,n,primary_correct,top2_correct,primary_acc,top2_acc
endproduct,,,,,
Agnostic,14,12,14,86%,100%
Cheese,7,7,7,100%,100%
"Chocolate, desserts, and confectionery",10,10,10,100%,100%
Cream and ice cream,10,10,10,100%,100%
Cross-cutting,15,11,15,73%,100%
Dairy,6,5,5,83%,83%
Eggs and egg proteins,7,7,7,100%,100%
Fish and seafood,9,9,9,100%,100%
Meat,9,9,9,100%,100%


,id,title,abstract,pillar,endproduct,primary_LLM,secondary_LLM,reasoning_LLM,primary_correct,either_correct
0,JP-2025522935-A,Pulsatile flow culture of mammary cell types f...,The apparatus and method for in vitro milk pro...,CM,Milk and milk proteins,Milk and milk proteins,Agnostic,The patent describes a bioreactor apparatus an...,True,True
1,US-20250176578-A1,PROCESS FOR PREPARING FERMENTED DAIRY PRODUCT ...,The invention relates to a process for prepari...,CC,Yoghurt and fermented dairy,Yoghurt and fermented dairy,Dairy,The patent explicitly describes a process for ...,True,True
2,US-20250098715-A1,METHOD FOR PREPARING OAT-BASED CONDIMENTS,The present invention relates to a method for ...,PB,"Spreads, sauces, and condiments","Spreads, sauces, and condiments",Agnostic,The patent explicitly targets oat-based savour...,True,True
3,US-20250241331-A1,ANIMAL-FREE SUBSTITUTE DAIRY FOOD PRODUCTS AND...,Methods of producing substitute dairy food pro...,F,Dairy,Dairy,Yoghurt and fermented dairy,The patent describes methods for producing ani...,True,True
4,EP-4514142-A1,GEL COMPOSITIONS AND THEIR USE IN SEAFOOD ANAL...,The present disclosure relates generally to ge...,CC,Fish and seafood,Fish and seafood,Agnostic,The patent explicitly describes gel compositio...,True,True
...,...,...,...,...,...,...,...,...,...,...
99,US-20250280859-A1,METHOD OF MAKING A SEAFOOD ANALOGUE,The present invention relates to a method of m...,PB,Fish and seafood,Fish and seafood,Cross-cutting,The patent explicitly describes a method of ma...,True,True
100,WO-2025233389-A1,STABILIZED LIQUID DEAMIDASE COMPOSITIONS,The invention provides methods for stabilizing...,PB,Cross-cutting,Cross-cutting,Agnostic,This patent describes a food-use enzyme compos...,True,True
101,EP-4633378-A1,CHOCOLATE COMPOSITION COMPRISING OMEGA-3-FATTY...,The present invention relates to a chocolate p...,PB,"Chocolate, desserts, and confectionery","Chocolate, desserts, and confectionery",Milk and milk proteins,The patent explicitly describes a chocolate pr...,True,True
102,EP-4498836-A1,VEGAN CREAMS AND ICE CREAMS,"The present invention relates to a vegan, non-...",PB,Cream and ice cream,Cream and ice cream,Dairy,The patent explicitly targets vegan/non-dairy ...,True,True


### 10. Save to Excel for Prompt Debugging

Order of working:
1. Create a new version folder in `1_prompt_debugging/`.
2. Copy in the previous prompt, label with the new version number, make updates.
3. Edit step 10 output directory and step 5 prompt/dataset selection.
4. Run steps 5–10.
5. Manually review results.
6. Document what changed and why. Repeat from step 1.

In [70]:
save_dir = Path(f"{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "top1_accuracy", "value": f"{comparison['primary_correct'].mean():.0%}", "n": n},
    {"metric": "top2_accuracy", "value": f"{comparison['either_correct'].mean():.0%}",  "n": n},
])

out_path = save_dir / f"{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")

Saved to v3\v3_claude-sonnet-4-6_results.xlsx


In [ ]:
# Create dataset of only rows where LLM got scope wrong, for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["correct_scope"], "id"]
incorrect_scope_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_scope_data